In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torch.utils.data import DataLoader

# --- THAY ĐỔI KÍCH THƯỚC BÓ THÀNH 10,000 ---
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    # Tăng batch_size từ 32 lên 10000 để thử nghiệm
    trn_dl = DataLoader(train, batch_size=10000, shuffle=True)
    
    val = FMNISTDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

# Khởi tạo lại dữ liệu và mô hình mới
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

# Tiến hành huấn luyện trong 5 Epochs
for epoch in range(5):
    print(f"Epoch đang chạy (Batch Size 10k): {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # Huấn luyện trên tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # Tính Accuracy tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # Đánh giá trên tập Validation
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
        val_accuracies.extend(val_is_correct) # Lưu ý: Theo ảnh gốc là sửa đổi tích lũy cho biểu đồ
    
    # Đồng bộ hóa đúng logic lưu trữ để vẽ đồ thị
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(np.mean(val_is_correct))

# --- TRỰC QUAN HÓA BIỂU ĐỒ KẾT QUẢ ---
epochs = np.arange(5) + 1
plt.figure(figsize=(10, 10))

# Đồ thị sai số (Loss) khi batch size = 10000
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss when batch size is 10000')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Đồ thị độ chính xác (Accuracy) khi batch size = 10000
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies[-5:], 'r', label='Validation accuracy') # Trích xuất đúng 5 epoch cuối phối hợp vẽ
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy when batch size is 10000')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()